Importing relevant libraries

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score
from sklearn.dummy import DummyClassifier

loading the data

In [2]:
df = pd.read_csv(r"C:\Users\Administrator\sales_202610292013.csv")
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

In [3]:
# Building Features (Adding Tenure and Days Between Purchases)
# We use a reference date(max date in the dataset) to calculate recency
max_date = df['transaction_date'].max()

In [6]:
# Group by customer to get the new features
rfm = df.groupby('customer_id').agg({
    'transaction_date': ['min','max','count'],
    'total_amount':'sum'})
rfm.columns = ['first_purchase','last_purchase','frequency','monetary']
rfm = rfm.reset_index()

Defining Churn

Time-Aware Split (To Avoid Data Leakage)

In [24]:
# Calculate tenure (days between first and last purchase)
rfm['tenure'] = (rfm['last_purchase'] - rfm['first_purchase']).dt.days
print(rfm)

      customer_id first_purchase last_purchase  frequency  monetary  tenure  \
0       C00000001     2024-04-05    2025-07-01          2    264.22     452   
1       C00000002     2023-07-11    2023-07-11          1    350.20       0   
2       C00000003     2024-11-02    2024-11-02          1     60.03       0   
3       C00000004     2023-10-05    2023-10-05          1    103.68       0   
4       C00000005     2024-04-09    2024-04-09          1     48.66       0   
...           ...            ...           ...        ...       ...     ...   
75794   C00119992     2025-07-09    2025-07-09          1      8.24       0   
75795   C00119993     2025-04-11    2025-04-11          1     29.84       0   
75796   C00119995     2024-06-04    2024-06-04          1     81.10       0   
75797   C00119997     2024-06-06    2026-03-01          2    140.40     633   
75798   C00120000     2024-05-12    2024-05-12          1     62.02       0   

       days_since_last_purchase  churned last_month

In [25]:
#Defining the features(We are not using days_since_last_purchase to avoid leakage)
features = ['frequency','monetary','tenure']
X = rfm[features]
y = rfm['churned']

In [26]:
# Standard 80/20 random split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [28]:
# Baseline Model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train,y_train)
base_pred = baseline.predict(X_test)
print(f"Baseline Accuracy is {accuracy_score(y_test,base_pred):.2f}")

Baseline Accuracy is 0.87


In [ ]:
# Random Forest Model(with class_weight to handle imbalance)
rf_model = RandomForestClassifier(random_state=42,class_weight='balanced')
rf_model.fit(X_train,y_train)
rf_pred = rf_model.predict(X_test)

print(f"\nRandom Forest Accuracy: {accuracy_score(y_test, rf_pred):.2f}")
print("\nClassification Report:\n", classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred):.2f}")